# Week 3 Live Session: Grading Fairly, and Your Nearest Neighbors

Today has two ideas that connect directly to what you already know:

1. A problem hiding in everything we've done so far: we've been grading our models using the same data they already saw. Today we fix that.
2. A brand new algorithm called K-Nearest Neighbors, which is probably closer to what you actually did in your internship. It uses math you already know from geometry class.


## Part 1: Are we grading fairly?

Quick question before we run anything: last week, we checked our logistic regression's accuracy. Which data did we check it on? The same data it trained on, or new data it had never seen?

(Think about it, then run the cell below.)


Right, the same data it trained on. Here's why that's a problem, using an analogy: imagine a student studies using the exact questions and answers from the real exam. Then they take that exact exam and score 100%. Did they actually learn the material, or did they just memorize the answer key?

We can't tell. And that's exactly the position we were in last week. Today we fix that with something called a **train/test split**: we hide some of our data away, train the model only on the rest, and then grade it only on the hidden part it has never seen.


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

df = pd.read_csv("data/session_current_temp_risk.csv")
df.head()

,current_mA,temperature_C,overheat_risk
0,7.4,25.9,0
1,19.5,26.7,0
2,41.0,70.2,1
3,6.1,38.7,0
4,15.3,39.8,0


This week's scenario combines both features you've already worked with: current (from Week 2) AND temperature (from Week 1). Together they decide whether a component is at overheat risk.

Let's split the data: 70% for training, 30% held back purely for testing.


In [13]:
X = df[["current_mA", "temperature_C"]]
y = df["overheat_risk"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("Training rows:", len(X_train))
print("Testing rows (held back, never trained on):", len(X_test))

Training rows: 49
Testing rows (held back, never trained on): 21


That's it. `train_test_split` just randomly separates rows into two piles. From now on, every model we build, we'll train on the training pile and grade honestly on the testing pile.


---
## Part 2: K-Nearest Neighbors (KNN)

This is likely close to what you actually did in your internship with KNN, just without the "why" behind it. Here's the plain idea:

> To classify a new point, look at the K closest points you already have answers for, and let them vote. Whatever label most of them have, that's your prediction.

That's the whole algorithm. No formula to fit, no line, no curve. Just "who's nearby, and what do they say?"

**The only real question is: how do we measure "closest"?** You already know the answer to this from geometry class. It's the distance formula:

`distance = sqrt((x1 - x2)^2 + (y1 - y2)^2)`

That's the Pythagorean theorem. Two points, a horizontal gap and a vertical gap, the hypotenuse between them is the distance. KNN just uses this same formula, treating our two features (current and temperature) as the x and y coordinates of a point.


We built this exact calculation by hand in the Excel file for today, sheet "KNN Distance and Voting". Let's switch over there for a minute; we'll pick one new point, calculate its distance to a handful of training points using this exact formula, sort them, and manually vote on the answer.

(Come back to this notebook once we've walked through that.)


Now let's do the same thing with actual code, on the full training set.


In [9]:
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

train_accuracy = accuracy_score(y_train, knn.predict(X_train))
test_accuracy = accuracy_score(y_test, knn.predict(X_test))

print("K=3, accuracy on TRAINING data:", train_accuracy)
print("K=3, accuracy on TESTING data (the honest score):", test_accuracy)

K=3, accuracy on TRAINING data: 0.9183673469387755
K=3, accuracy on TESTING data (the honest score): 0.8571428571428571


### Now the important demo: what happens at K=1?

Think back to Part 1's exam analogy. If K=1, and we check accuracy using the training data itself, what is a training point's single nearest neighbor always going to be?


In [14]:
knn1 = KNeighborsClassifier(n_neighbors=1)
knn1.fit(X_train, y_train)

train_acc_k1 = accuracy_score(y_train, knn1.predict(X_train))
test_acc_k1 = accuracy_score(y_test, knn1.predict(X_test))

print("K=1, accuracy on TRAINING data:", train_acc_k1)
print("K=1, accuracy on TESTING data:", test_acc_k1)

K=1, accuracy on TRAINING data: 1.0
K=1, accuracy on TESTING data: 0.8095238095238095


Its own nearest neighbor is itself, distance zero. So K=1 will always score 100% on training data, no matter how good the model actually is. Notice its TEST accuracy is worse than K=3's. That's called **overfitting**: the model memorized the training data instead of learning the real pattern, so it does badly on anything new.

This is the exact same "studied off the answer key" problem from Part 1, just visible in a single number now.


### Trying a few different values of K

Let's see how training and testing accuracy change as K goes up.


In [15]:
for k in [1, 3, 5, 9]:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_train, y_train)
    tr = accuracy_score(y_train, knn_k.predict(X_train))
    te = accuracy_score(y_test, knn_k.predict(X_test))
    print(f"K={k}:  training accuracy={tr:.2f}   testing accuracy={te:.2f}")

K=1:  training accuracy=1.00   testing accuracy=0.81
K=3:  training accuracy=0.92   testing accuracy=0.86
K=5:  training accuracy=0.96   testing accuracy=0.86
K=9:  training accuracy=0.90   testing accuracy=0.86


Notice training accuracy generally goes down a bit as K increases, but testing accuracy becomes more trustworthy and stable. A very small K memorizes. A reasonable K generalizes better. Picking a good K is part of the skill.


### Confusion matrix on the TEST set

Same evaluation tool from last week, but now on data the model never saw, which is the honest way to use it.


In [12]:
predicted_test = knn.predict(X_test)
cm = confusion_matrix(y_test, predicted_test)
cm_table = pd.DataFrame(cm,
    index=["Actual: Safe", "Actual: Risk"],
    columns=["Predicted: Safe", "Predicted: Risk"])
cm_table

,Predicted: Safe,Predicted: Risk
Actual: Safe,7,2
Actual: Risk,1,11


---
## Recap

- Grading a model on the same data it trained on can lie to you. A train/test split fixes that by holding back data the model never sees, for honest grading.
- KNN classifies a new point by finding its K closest neighbors (using the same distance formula as the Pythagorean theorem) and letting them vote.
- K=1 will always look perfect on training data, since a point's nearest neighbor is always itself. That's a textbook example of overfitting.
- Picking a reasonable K, and always checking test accuracy (not training accuracy), is how you catch this.

Homework this week is a new, fun scenario: predicting whether a laptop's cooling fan turns on, using the exact same KNN + train/test split pattern.
